# データ検証レシピ

## 商品単価が自動取得できない！データ検証をして改善案を考えよう

> 架空データを使って検証します。

## 検証すること

単価紐づけのための現行キーは「商品コード」「提供先コード」「バリエーション」「加工区分」「供給ルート」です。
このキーを使って、単価マスタから商品単価を取得していますが、「供給ルート」が不安定で、取得できないケースが多発しています。
不安定な「供給ルート」はキーから外しても単価が取得できるのではないか。
キーを減らしても価格が一意になるかを検証してみます。

## 1. データの読み込み


In [ ]:
import pandas as pd

df = pd.read_csv(
    "price.csv",
    encoding="cp932",
)

print(df.shape)
display(df.head())

In [ ]:
# 読み込んだデータの列名を確認してみます

print(list(df.columns))

## 2. 健康診断


In [ ]:
# 空白数を確認します
print(df.isna().sum())

#　データ型を確認します
print(df.dtypes)


## 3. 空欄を整理する

- 「NaN」は文字が入っているのではなく、空欄を意味しています。数えるときに空欄を見落とさないよう、空欄のままでよいものは「（空）」という文字に置き換えます。
- 今回の目的は、商品ごとの価格を正しく確認することです。「商品コード」「提供先コード」「登録価格」は大事な項目なので、空欄の行は別に分けて確認します。


In [ ]:
# 空欄に「(空)」を入れてdf2というデータフレームを作成します
df2 = df.fillna({"バリエーション": "(空)", "加工区分": "(空)", "供給ルート": "(空)"})

# 商品コード、提供先コード、登録価格のいずれかが空欄の行をdf2から取り出して、checkというデータフレームを作成します
check = df2[df2["商品コード"].isna()| df2["提供先コード"].isna()| df2["登録価格"].isna()]

# 大事な項目がすべて入っている行のみのデータフレームを作成します
ok = df2[df2["商品コード"].notna() & df2["提供先コード"].notna() & df2["登録価格"].notna()]

print("df:",len(df))
print("ok:",len(ok))
print("check:",len(check))
display(check)


## 4. 商品コードと提供先コードの組み合わせを確認する

In [ ]:
key = ["商品コード", "提供先コード"]

same_key = ok[ok.duplicated(key, keep=False)]

print(len(same_key))
display(same_key)

## 5. 現行キーで価格が一意か確認する

In [ ]:
current_key = ["商品コード","提供先コード","バリエーション","加工区分","供給ルート"]

current_price_count = (ok.groupby(current_key)["登録価格"].nunique())

current_check = current_price_count[current_price_count > 1]

print(len(current_check))
display(current_check)

## 6. 供給ルートを外して価格が一意か確認する

In [ ]:
candidate_key = [
    "商品コード",
    "提供先コード",
    "バリエーション",
    "加工区分",
]

candidate_price_count = (
    ok.groupby(candidate_key)["登録価格"]
    .nunique()
)

candidate_check = candidate_price_count[
    candidate_price_count > 1
]

print(len(candidate_check))
display(candidate_check)

## 7. 要確認の明細を見る

In [ ]:
problem_keys = (
    candidate_check
    .reset_index()[candidate_key]
)

problem_detail = ok.merge(
    problem_keys,
    on=candidate_key,
    how="inner",
)

display(problem_detail)

## 8. 完全に同じ行を確認する

In [ ]:
duplicate_rows = df[
    df.duplicated(keep=False)
]

print(len(duplicate_rows))
display(duplicate_rows)

## 9. 要確認データをCSVに出す

In [ ]:
check.to_csv(
    "check_blank.csv",
    index=False,
    encoding="utf-8-sig",
)

problem_detail.to_csv(
    "check_price.csv",
    index=False,
    encoding="utf-8-sig",
)

## 10. 結論

- 現行キーで価格が一意でない組：0組
- 供給ルートを外すと価格が一意でない組：3組
- 該当する価格差を業務確認し、複数期間のデータで再検証してからキー変更を判断します。